# Fine-tune Laya for Hangman (Kaggle or Colab, GPU)

Setup: Accelerator = GPU (T4), Internet = On. Upload `train.jsonl.gz` (from the repo's `data/` folder) to the notebook's working files (Colab: Files panel) or add it as a Kaggle dataset, then **Run All**. Cell 3 is a dry run: check it reports 0 skipped rows. Cell 4 trains (about an hour on a T4, an unmeasured estimate). See `docs/FINETUNE.md`.

In [ ]:
!pip install -q laya==0.4.6
import torch; print('GPU available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on a GPU accelerator first'

In [ ]:
import glob, os, shutil
cands = glob.glob('train.jsonl.gz') + glob.glob('/kaggle/input/**/train.jsonl.gz', recursive=True) + glob.glob('/content/train.jsonl.gz')
assert cands, 'Upload train.jsonl.gz first (see the instructions above)'
shutil.copy(cands[0], 'train.jsonl.gz')
!gunzip -kf train.jsonl.gz
!wc -l train.jsonl   # expect 11292

In [ ]:
# Dry run: builds items with the real tokenizer, no training. 'skipped' must be 0.
!laya-train --data train.jsonl --base english --max-len 512 --head-max-len 192 --dry-run

In [ ]:
# Training (long). Output: ./laya-hangman-ft
!laya-train --data train.jsonl --base english --out ./laya-hangman-ft --loss soft-ce --shuffle-options --epochs 4 --max-len 512 --head-max-len 192

In [ ]:
# Package the checkpoint for download (about 1-2 GB)
!zip -qr laya-hangman-ft.zip laya-hangman-ft && ls -lh laya-hangman-ft.zip

Download `laya-hangman-ft.zip` (Kaggle: Output tab; Colab: Files panel), unzip it on your Mac, then evaluate with:

`LAYA_MODEL=/path/to/laya-hangman-ft .venv/bin/python -m laya_hangman.evaluate --split eval --deciders baseline laya --limit 40 --progress`